<a href="https://colab.research.google.com/github/irenetobby/streamlit_fastapi_bank/blob/main/fastapi_bank_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from sklearn.ensemble import RandomForestClassifier
import pickle
import numpy as np

# Create a dummy classifier for demonstration
# In a real scenario, you would load your pre-trained model here
classifier = RandomForestClassifier(random_state=42)

# Create some dummy data and labels to 'train' the dummy classifier
X_dummy = np.random.rand(10, 4) # 10 samples, 4 features
y_dummy = np.random.randint(0, 2, 10) # 10 binary labels

classifier.fit(X_dummy, y_dummy)

# Save the dummy classifier to 'Classifier.pkl'
with open('Classifier.pkl', 'wb') as file:
    pickle.dump(classifier, file)

print("Dummy Classifier.pkl created successfully.")

Dummy Classifier.pkl created successfully.


In [ ]:
from fastapi import FastAPI
import uvicorn
import numpy as np
import pickle
import pandas as pd
from pydantic import BaseModel
import threading

# Define the BankNote class directly in this file
class BankNote(BaseModel):
    variance: float
    skewness: float
    curtosis: float
    entropy: float

# creating the instance
app = FastAPI()
pickle_in = open('Classifier.pkl', 'rb')
# make the classifier model from pickle
classifier = pickle.load(pickle_in)
# index route for opening the local host
@app.get('/')
def index():
    return {'message': 'Hello, Welcome'}
@app.get('/Welcome')
def get_name(name: str):
    return {'Welcome to sample Fast API Deployment': f'{name}'}
@app.post('/predict')
def predict_banknote(data: BankNote): # the class created in BankNotes.py, data created in JSON form, now convert to dictionary
    data = data.dict()
    variance = data['variance']
    skewness = data['skewness']
    curtosis = data['curtosis']
    entropy = data['entropy']

    prediction = classifier.predict([[variance, skewness, curtosis, entropy]])
    if prediction[0] > 0.5:
        prediction = 'The note is Fake'
    else:
        prediction = 'The note is original'
    return {'prediction': prediction}

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

if __name__ == "__main__":
    # Start Uvicorn in a separate thread
    uvicorn_thread = threading.Thread(target=run_uvicorn)
    uvicorn_thread.daemon = True  # Allow the program to exit even if the thread is running
    uvicorn_thread.start()
    print("Uvicorn server started in a separate thread at http://0.0.0.0:8000")
    # Keep the main thread alive to allow the server thread to run
    # In a Colab environment, the cell will complete execution, but the thread will continue.
    # You might want to add a blocking call like input() if you need to manually stop the server,
    # but for continuous background operation, this is usually sufficient.
